# 01 - RAG environment check: local GPU inference

A personal practice adaptation of the supplied NVIDIA DLI **LLM Services and AI Foundation Models** material. This is not an official course notebook.

**Learning objective:** replace a hosted chat client with a model running inside this notebook's Python process, and verify that local generation preserves the prompt and runs on the GPU. No `ChatNVIDIA`, `llm_client` service, NVIDIA endpoint, or API key is required.

Model files can be downloaded once from Hugging Face in an opt-in setup step. Loading uses local files only, and inference does not send prompts to a hosted model.

## What changes from the course?

A hosted client delegates model loading, GPU memory management, and generation to a server. Here, the notebook owns those responsibilities: the CPU prepares token IDs, then the GPU runs the model on those IDs. Local inference keeps the generation workload and prompts on your workstation, but each workstation must have enough memory and compute. A hosted service can instead share expensive hardware across users and support lightweight clients. Neither deployment removes the need to choose a suitable model and control its resource use.

| Course example | Local adaptation |
| --- | --- |
| `ChatNVIDIA` and a service URL | Transformers model and tokenizer in this Python kernel |
| `nvidia/nemotron-3.5-lightning-30b-a3b` | `Qwen/Qwen3-4B`, chosen for a short, single-GPU exercise |
| `max_completion_tokens` | `max_new_tokens`, excluding the input prompt |
| Server-side thinking options | A model-specific chat-template option; disabled for the first checkpoint |
| `llm.stream()` yields message chunks | Later: a local text streamer; decoded strings are not `AIMessageChunk` objects |
| Service response metadata | Locally inspected token counts, device, and eventually timing |

The 30B-A3B model's active parameter count is not its total weight-storage requirement. This exercise deliberately changes models rather than assuming those weights fit in 24 GB of VRAM. Qwen3-4B needs roughly 8 GB for BF16 weights alone, plus memory for the KV cache and temporary tensors; free VRAM and prompt length still matter. Model behavior and answers will differ from the course model.

## Learning milestones

1. **Now:** load a local model and generate one non-streaming reply on the GPU.
2. **After the first checkpoint:** stream decoded text with a local streamer, including error and interruption handling.
3. **Then:** enable Qwen3 thinking and distinguish reasoning text, final text, and locally measured metadata.
4. **Later:** connect the working local inference path to LangChain.

**Provided boilerplate:** dependency instructions, CUDA checks, model paths, optional model download, model/tokenizer loading, prompt preparation, and assertions.

**Your learning-critical work:** the local generation operation in milestone 1. This notebook intentionally stops at that operation and its checkpoint; later milestones are not implemented yet.

## 1. Prepare the runtime

Open the repository's dev container and select its Python kernel. It uses `nvcr.io/nvidia/pytorch:24.03-py3` with GPU access; the target workstation has a 24 GB RTX 4090. Unlike the supplied course's CPU-only client environment, this notebook requires a CUDA-capable local PyTorch runtime and has no CPU or hosted-service fallback.

From the **repository root in the container terminal**, install the course dependencies:

```bash
python -m pip install -r courses/building-rag-agents-with-llms/requirements.txt
```

Restart the notebook kernel after installation. Keep the container's PyTorch/CUDA installation rather than installing a separate `torch` wheel. The pinned Transformers release supports Qwen3; versions below 4.51.0 do not.

Run the setup cells in order. If CUDA is unavailable, inspect the selected kernel and run `nvidia-smi` in the container terminal before changing drivers or packages.

In [1]:
import sys
from importlib.metadata import version
from pathlib import Path

import torch
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer

cwd = Path.cwd().resolve()
repo_root = next(
    (path for path in (cwd, *cwd.parents)
     if (path / "courses/building-rag-agents-with-llms/notebooks").is_dir()),
    None,
)
if repo_root is None:
    raise RuntimeError("Start the notebook kernel inside the repository checkout.")
course_dir = repo_root / "courses/building-rag-agents-with-llms"

print("Python:", sys.version.split()[0], "|", sys.executable)
print("PyTorch:", torch.__version__, "| CUDA runtime:", torch.version.cuda)
print("Transformers:", version("transformers"))
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select the GPU-enabled dev-container kernel.")

device = torch.device("cuda:0")
torch.cuda.set_device(device)
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This checkpoint uses BF16 weights; select the intended BF16-capable GPU.")
model_dtype = torch.bfloat16
free_bytes, total_bytes = torch.cuda.mem_get_info(device)
print("GPU:", torch.cuda.get_device_name(device))
print(f"VRAM: {free_bytes / 2**30:.1f} GiB free / {total_bytes / 2**30:.1f} GiB total")

probe = torch.ones((2, 2), device=device)
assert (probe @ probe).tolist() == [[2.0, 2.0], [2.0, 2.0]]
del probe


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.10.12 | /usr/bin/python
PyTorch: 2.3.0a0+40ec155e58.nv24.03 | CUDA runtime: 12.4
Transformers: 4.52.4
GPU: NVIDIA GeForce RTX 4090
VRAM: 22.4 GiB free / 24.0 GiB total


## 2. Use local model files

The default directory is `courses/building-rag-agents-with-llms/models/Qwen3-4B/`. Either point `MODEL_DIR` at an existing complete Qwen3-4B snapshot, or explicitly set `DOWNLOAD_MODEL = True` below to download it once. The download requires internet access and roughly 8 GB of disk space for weights, plus supporting files; it does not start hosted inference. The selected public model does not require an API key.

The `models/` directory is ignored by Git. Keep the model's configuration, tokenizer, safetensors shards, and shard index together. Leave downloads disabled after setup. Missing or incomplete files must produce an error rather than silently switching to a remote model.

Model-specific guidance: [Qwen3-4B model card](https://huggingface.co/Qwen/Qwen3-4B). Different model families may require different templates, precision, and generation settings.

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B"
MODEL_DIR = course_dir / "models" / "Qwen3-4B"
DOWNLOAD_MODEL = False

if DOWNLOAD_MODEL:
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=MODEL_DIR,
        allow_patterns=["*.json", "*.safetensors", "*.txt", "*.jinja", "LICENSE*", "README.md"],
        token=False,
    )
else:
    print("Download disabled; expecting existing model files at:", MODEL_DIR)


## 3. Load the model onto one GPU

The loader reads safetensors from disk without executing repository-supplied Python code. BF16 is an explicit choice for this model and the target RTX 4090: about two bytes per weight instead of four for FP32. Token IDs remain integers; they must not be cast to the model's floating-point dtype. The model is loaded through CPU memory before moving to `cuda:0`, so sufficient host RAM is also required.

This setup deliberately avoids automatic CPU offload. Run the loading cell once per kernel; restart the kernel before reloading or switching large models. An out-of-memory error means the local configuration needs attention, not that inference should silently move elsewhere.

In [ ]:
if not (MODEL_DIR / "config.json").is_file():
    raise FileNotFoundError(
        f"No model configuration at {MODEL_DIR}. Provide a complete local snapshot "
        "or explicitly enable the download cell, then run it first."
    )

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
    trust_remote_code=False,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
    trust_remote_code=False,
    use_safetensors=True,
    torch_dtype=model_dtype,
    attn_implementation="sdpa",
).to(device)
model.eval()

assert all(parameter.device == device for parameter in model.parameters())
assert next(model.parameters()).dtype == model_dtype
print("Loaded local model:", MODEL_ID)
print("Model device:", model.device, "| weight dtype:", model.dtype)


## 4. Prepare one chat prompt

The tokenizer's chat template replaces the server's message formatting. For this first milestone, thinking is explicitly disabled so that we can check a short visible reply before tackling streaming. Sampling follows the model card's non-thinking defaults rather than copying the course client's `temperature=0`.

Later, Qwen3 thinking uses a `<think>...</think>` text convention, not `chunk.additional_kwargs['reasoning_content']`. Native Transformers generation has no direct equivalent of the course server's `reasoning_budget` argument. `max_new_tokens` limits all generated tokens, including reasoning when enabled; a short budget may end before a final answer.

In [ ]:
messages = [{"role": "user", "content": "Tell me about yourself! 2 sentences."}]
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
model_inputs = tokenizer(
    prompt, return_tensors="pt", add_special_tokens=False,
).to(device)

generation_kwargs = {
    "max_new_tokens": 256,
    "do_sample": True,
    "temperature": 0.7,
    "top_p": 0.8,
    "top_k": 20,
    "pad_token_id": tokenizer.eos_token_id,
    "return_dict_in_generate": False,
}

print("Input shape [batch, prompt tokens]:", tuple(model_inputs["input_ids"].shape))
print("Input dtype:", model_inputs["input_ids"].dtype, "| device:", model_inputs["input_ids"].device)


## 5. Your turn: generate locally

**Concept:** a decoder-only model extends the input token sequence. Unlike a chat client's final message, the returned sequence includes both the original prompt and newly generated tokens. The model and inputs must stay on the same GPU. Evaluation mode was set during loading; the decorator below also disables gradient tracking for this inference call.

**Prediction:** before running, record the expected output shape relative to the printed input shape and the 256-token generation limit.

Prediction: _write your prediction here_.

**Implement only `generate_local`:** use the local model's generation API with the prepared input and generation-setting dictionaries. Return the complete token-ID tensor, keeping the prompt intact; do not decode it or move it to the CPU inside the function. The checkpoint handles those observations for you.

In [ ]:
@torch.inference_mode()
def generate_local(model, model_inputs, generation_kwargs):
    """Return token IDs with shape [batch, prompt tokens + new tokens]."""
    # TODO: Perform local generation using the supplied model and both dictionaries.
    raise NotImplementedError("Implement the local generation call, then run the checkpoint.")


## 6. Checkpoint - then stop

Run this cell after implementing the function. It checks the output tensor's batch size, device, integer dtype, unchanged prompt prefix, and generation limit, then decodes only the new tokens. A correct run prints a visible reply and locally computed metadata; exact wording is not fixed, and two sentences are a prompt request rather than an assertion.

An unimplemented function deliberately raises `NotImplementedError`. Share your function and the checkpoint result for review before adding streaming or thinking-mode code. Clear notebook outputs before committing.

In [ ]:
output_ids = generate_local(model, model_inputs, generation_kwargs)
input_ids = model_inputs["input_ids"]
prompt_length = input_ids.shape[1]

assert isinstance(output_ids, torch.Tensor), "Return token IDs, not decoded text or a result object."
assert output_ids.ndim == 2 and output_ids.shape[0] == input_ids.shape[0] == 1
assert output_ids.device == input_ids.device == model.device, "Keep generation on the model's GPU."
assert output_ids.dtype == input_ids.dtype == torch.long, "Token IDs must remain integers."
assert torch.equal(output_ids[:, :prompt_length], input_ids), "The output must retain the prompt prefix."

new_ids = output_ids[0, prompt_length:]
assert 0 < new_ids.numel() <= generation_kwargs["max_new_tokens"], "Check the new-token limit."
reply = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
assert reply, "No visible reply was generated; inspect the token IDs and stopping settings."

print("[ RESPONSE ]", reply)
print("[ LOCAL METADATA ]", {
    "prompt_tokens": prompt_length,
    "generated_tokens": new_ids.numel(),
    "device": str(output_ids.device),
    "token_dtype": str(output_ids.dtype),
})
if new_ids.numel() == generation_kwargs["max_new_tokens"]:
    print("The token limit was reached; inspect whether the reply was cut short.")
